In [1]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# File path
file_path = "Project4_Dataset.xlsx"

# Load worksheets
customers_raw = pd.read_excel(file_path, sheet_name="Customers")
bookings_raw = pd.read_excel(file_path, sheet_name="Bookings")
service_requests_raw = pd.read_excel(
    file_path,
    sheet_name="Service_Requests"
)

# Create working copies
customers = customers_raw.copy()
bookings = bookings_raw.copy()
service_requests = service_requests_raw.copy()

print("Customers:", customers.shape)
print("Bookings:", bookings.shape)
print("Service Requests:", service_requests.shape)

Customers: (1500, 50)
Bookings: (3500, 64)
Service Requests: (4500, 60)


In [2]:
datasets = {
    "Customers": customers,
    "Bookings": bookings,
    "Service Requests": service_requests
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Duplicate rows: {df.duplicated().sum():,}")
    print(f"Total missing values: {df.isna().sum().sum():,}")


Customers
--------------------------------------------------
Rows: 1,500
Columns: 50
Duplicate rows: 0
Total missing values: 2,529

Bookings
--------------------------------------------------
Rows: 3,500
Columns: 64
Duplicate rows: 0
Total missing values: 3,473

Service Requests
--------------------------------------------------
Rows: 4,500
Columns: 60
Duplicate rows: 0
Total missing values: 10,657


In [3]:
primary_keys = {
    "Customers": "customer_id",
    "Bookings": "booking_id",
    "Service Requests": "ticket_id"
}

for name, df in datasets.items():
    key = primary_keys[name]

    print(f"\n{name} — {key}")
    print("-" * 50)
    print(f"Missing primary keys: {df[key].isna().sum():,}")
    print(
        f"Duplicated key rows: "
        f"{df.loc[df[key].notna(), key].duplicated(keep=False).sum():,}"
    )
    print(
        f"Duplicated unique keys: "
        f"{df.loc[df[key].notna() & df[key].duplicated(keep=False), key].nunique():,}"
    )


Customers — customer_id
--------------------------------------------------
Missing primary keys: 19
Duplicated key rows: 0
Duplicated unique keys: 0

Bookings — booking_id
--------------------------------------------------
Missing primary keys: 0
Duplicated key rows: 119
Duplicated unique keys: 59

Service Requests — ticket_id
--------------------------------------------------
Missing primary keys: 0
Duplicated key rows: 143
Duplicated unique keys: 71


In [4]:
for name, df in datasets.items():
    key = primary_keys[name]

    print(f"\n{name}")
    print("-" * 50)

    print("is_test_record:")
    print(df["is_test_record"].value_counts(dropna=False))

    print(f"\nRows with missing {key}:")
    print(df[df[key].isna()].shape[0])


Customers
--------------------------------------------------
is_test_record:
is_test_record
N    1492
Y       8
Name: count, dtype: int64

Rows with missing customer_id:
19

Bookings
--------------------------------------------------
is_test_record:
is_test_record
N    3466
Y      34
Name: count, dtype: int64

Rows with missing booking_id:
0

Service Requests
--------------------------------------------------
is_test_record:
is_test_record
N    4449
Y      51
Name: count, dtype: int64

Rows with missing ticket_id:
0


In [5]:
customers = customers[
    customers["is_test_record"].str.upper().ne("Y")
].copy()

bookings = bookings[
    bookings["is_test_record"].str.upper().ne("Y")
].copy()

service_requests = service_requests[
    service_requests["is_test_record"].str.upper().ne("Y")
].copy()

print("Customers after removing test records:", customers.shape)
print("Bookings after removing test records:", bookings.shape)
print(
    "Service Requests after removing test records:",
    service_requests.shape
)

Customers after removing test records: (1492, 50)
Bookings after removing test records: (3466, 64)
Service Requests after removing test records: (4449, 60)


In [6]:
cleaning_log = []

# Remove rows with missing primary keys
for name, df, key in [
    ("Customers", customers, "customer_id"),
    ("Bookings", bookings, "booking_id"),
    ("Service Requests", service_requests, "ticket_id")
]:
    missing_count = df[key].isna().sum()

    cleaning_log.append({
        "table": name,
        "issue": f"Missing {key}",
        "action": "Removed",
        "rows_affected": missing_count
    })

    print(f"{name}: {missing_count} rows removed")

Customers: 18 rows removed
Bookings: 0 rows removed
Service Requests: 0 rows removed


In [7]:
customers = customers.dropna(subset=["customer_id"]).copy()

bookings = bookings.dropna(subset=["booking_id"]).copy()

service_requests = service_requests.dropna(
    subset=["ticket_id"]
).copy()

print("Customers:", customers.shape)
print("Bookings:", bookings.shape)
print("Service Requests:", service_requests.shape)

Customers: (1474, 50)
Bookings: (3466, 64)
Service Requests: (4449, 60)


In [8]:
duplicate_bookings = bookings[
    bookings["booking_id"].duplicated(keep=False)
].sort_values(
    by=["booking_id", "record_version"]
)

duplicate_tickets = service_requests[
    service_requests["ticket_id"].duplicated(keep=False)
].sort_values(
    by=["ticket_id", "record_version"]
)

print(
    "Duplicated booking IDs:",
    duplicate_bookings["booking_id"].nunique()
)

print(
    "Duplicated ticket IDs:",
    duplicate_tickets["ticket_id"].nunique()
)

Duplicated booking IDs: 59
Duplicated ticket IDs: 71


In [9]:
booking_review_columns = [
    "booking_id",
    "record_version",
    "audit_timestamp",
    "data_quality_flag",
    "manual_override_flag",
    "customer_id",
    "property_id",
    "booking_date",
    "check_in_date",
    "check_out_date",
    "status",
    "total_revenue",
    "guest_satisfaction_score"
]

duplicate_bookings[booking_review_columns].head(20)

,booking_id,record_version,audit_timestamp,data_quality_flag,manual_override_flag,customer_id,property_id,booking_date,check_in_date,check_out_date,status,total_revenue,guest_satisfaction_score
62,BKG_000063,1,2024-10-12 03:43,Potential Test Record,N,CUST_00350,P043,15-Apr-2023,15-Apr-2023,18-Apr-2023,No Show,0.00,NaN
112,BKG_000063,2,03/03/2025 18:49,Date Format Issue,N,CUST_00546,P020,02/13/2025,2025/03/07,03/12/2025,Completed,"3,358.23",4.70
104,BKG_000064,1,08-Mar-2025 06:21,Potential Test Record,N,CUST_00256,P031,26-Dec-2022,17-Feb-2023,02/18/2023,Completed,254.55,4.00
63,BKG_000064,2,2025-05-09 18:41,Potential Test Record,N,CUST_01198,P018,2024-02-13,04/14/2024,2024/04/15,Cancelled,0.00,NaN
141,BKG_000142,2,2025-05-11 10:20,Manual Review,N,CUST_00110,P010,2023-08-12,17-Sep-2023,2023-09-21,Completed,"1,349.14",NaN
165,BKG_000142,2,11-Apr-2025 04:29,Date Format Issue,N,CUST_00865,P006,2023/10/29,10-Dec-2023,2023/12/12,Cancelled,0.00,2.50
262,BKG_000218,1,2025-04-13 09:47,Date Format Issue,N,CUST_00758,P034,2024/01/02,04-Mar-2024,2024/03/08,Completed,"1,274.21",4.70
217,BKG_000218,2,2024-11-21 17:18,OK,N,CUST_00468,P026,2024/01/10,01/10/2024,2024-01-12,Completed,746.60,5.00
251,BKG_000252,1,10/13/2024 02:08,Duplicate-Like,N,CUST_00469,P032,2023/05/11,04-Aug-2023,2023-08-10,Completed,"2,760.82",3.40
305,BKG_000252,1,03/18/2025 05:20,Date Format Issue,N,CUST_00396,P011,14-Feb-2025,02/15/2025,2025/02/18,Completed,"3,839.10",3.70


In [10]:
ticket_review_columns = [
    "ticket_id",
    "record_version",
    "audit_timestamp",
    "data_quality_flag",
    "manual_override_flag",
    "booking_id",
    "customer_id",
    "property_id",
    "issue_type",
    "ticket_status",
    "open_timestamp",
    "resolution_time_hours",
    "sla_breach_flag",
    "customer_sentiment",
    "satisfaction_after_resolution"
]

duplicate_tickets[ticket_review_columns].head(20)

,ticket_id,record_version,audit_timestamp,data_quality_flag,manual_override_flag,booking_id,customer_id,property_id,issue_type,ticket_status,open_timestamp,resolution_time_hours,sla_breach_flag,customer_sentiment,satisfaction_after_resolution
205,TKT_000160,1,10/05/2024 08:49,OK,N,BKG_003022,C-00179,P014,Billing Question,Closed,2025-04-30 19:10,35.80,N,Neutral,3.80
159,TKT_000160,2,01/19/2025 10:53,OK,N,BKG_002388,CUST_00542,P056,Noise Complaint,Closed,14-Jan-2023 15:15,3.40,N,Negative,4.60
269,TKT_000270,1,2025-04-12 23:24,Duplicate-Like,N,BKG_003093,CUST_00150,P050,WiFi Issue,Closed,02/04/2023 03:15,17.50,N,Positive,3.50
301,TKT_000270,2,01/18/2025 13:36,Duplicate-Like,N,NaN,CUST_00771,P016,Housekeeping Delay,Open,06-Jul-2024 05:10,NaN,N,Positive,NaN
328,TKT_000329,3,15-Nov-2024 23:51,OK,N,BKG_000042,CUST_00253,P042,Check-in Delay,Closed,07/02/2024 02:15,37.10,Y,Neutral,5.00
420,TKT_000329,3,19-Dec-2024 10:28,OK,N,BKG_000934,CUST_01219,P010,Maintenance,Closed,02/23/2025 06:15,12.10,Y,Very Negative,NaN
337,TKT_000338,2,05-May-2025 13:03,Potential Test Record,N,BKG_003412,CUST_01359,P049,Check-in Delay,Open,11-Nov-2024 18:05,NaN,Y,Very Negative,NaN
412,TKT_000338,2,03/21/2025 23:25,Potential Test Record,N,BKG_001594,CUST_00815,P042,Food & Beverage,Closed,03/30/2023 07:15,0.50,N,Neutral,5.00
427,TKT_000428,3,23-Mar-2025 14:10,OK,N,BKG_003483,cust_01147,P036,Housekeeping Delay,Open,2023-09-27 20:10,NaN,N,Negative,NaN
437,TKT_000428,3,2025-03-20 14:19,Missing Field,N,BKG_001344,CUST_00128,P010,Food & Beverage,Closed,17-Dec-2024 18:45,0.50,N,Positive,4.50


In [11]:
# Standardize audit timestamps
customers["audit_timestamp"] = pd.to_datetime(
    customers["audit_timestamp"],
    errors="coerce",
    format="mixed"
)

bookings["audit_timestamp"] = pd.to_datetime(
    bookings["audit_timestamp"],
    errors="coerce",
    format="mixed"
)

service_requests["audit_timestamp"] = pd.to_datetime(
    service_requests["audit_timestamp"],
    errors="coerce",
    format="mixed"
)

print(
    "Invalid customer audit timestamps:",
    customers["audit_timestamp"].isna().sum()
)

print(
    "Invalid booking audit timestamps:",
    bookings["audit_timestamp"].isna().sum()
)

print(
    "Invalid service request audit timestamps:",
    service_requests["audit_timestamp"].isna().sum()
)


Invalid customer audit timestamps: 0
Invalid booking audit timestamps: 0
Invalid service request audit timestamps: 0


In [12]:
quality_rank = {
    "OK": 0,
    "Date Format Issue": 1,
    "Duplicate-Like": 2,
    "Manual Review": 3,
    "Missing Field": 4,
    "Potential Test Record": 5
}

In [13]:
booking_critical_columns = [
    "customer_id",
    "property_id",
    "booking_date",
    "check_in_date",
    "check_out_date",
    "status",
    "num_rooms",
    "nightly_rate",
    "total_revenue"
]

# Identify records involved in duplicate-ID conflicts
bookings["duplicate_id_flag"] = np.where(
    bookings["booking_id"].duplicated(keep=False),
    "Y",
    "N"
)

# Data-quality rank
bookings["_quality_rank"] = (
    bookings["data_quality_flag"]
    .map(quality_rank)
    .fillna(99)
)

# Number of missing critical fields
bookings["_critical_missing"] = (
    bookings[booking_critical_columns]
    .isna()
    .sum(axis=1)
)

# Make sure record version is numeric
bookings["record_version"] = pd.to_numeric(
    bookings["record_version"],
    errors="coerce"
)

In [14]:
booking_critical_columns = [
    "customer_id",
    "property_id",
    "booking_date",
    "check_in_date",
    "check_out_date",
    "status",
    "num_rooms",
    "nightly_rate",
    "total_revenue"
]

# Identify records involved in duplicate-ID conflicts
bookings["duplicate_id_flag"] = np.where(
    bookings["booking_id"].duplicated(keep=False),
    "Y",
    "N"
)

# Data-quality rank
bookings["_quality_rank"] = (
    bookings["data_quality_flag"]
    .map(quality_rank)
    .fillna(99)
)

# Number of missing critical fields
bookings["_critical_missing"] = (
    bookings[booking_critical_columns]
    .isna()
    .sum(axis=1)
)

# Make sure record version is numeric
bookings["record_version"] = pd.to_numeric(
    bookings["record_version"],
    errors="coerce"
)

In [15]:
bookings_ranked = bookings.sort_values(
    by=[
        "booking_id",
        "_quality_rank",
        "_critical_missing",
        "record_version",
        "audit_timestamp"
    ],
    ascending=[
        True,    # booking_id
        True,    # lower quality rank is better
        True,    # fewer missing fields is better
        False,   # higher record version is better
        False    # newer audit timestamp is better
    ],
    na_position="last"
)

bookings_clean = bookings_ranked.drop_duplicates(
    subset=["booking_id"],
    keep="first"
).copy()

In [16]:
booking_duplicates_removed = len(bookings) - len(bookings_clean)

print("Bookings before deduplication:", len(bookings))
print("Bookings after deduplication:", len(bookings_clean))
print("Duplicate booking rows removed:", booking_duplicates_removed)
print(
    "Unique booking IDs:",
    bookings_clean["booking_id"].nunique()
)

Bookings before deduplication: 3466
Bookings after deduplication: 3406
Duplicate booking rows removed: 60
Unique booking IDs: 3406


In [17]:
 ticket_critical_columns = [
    "booking_id",
    "customer_id",
    "property_id",
    "issue_type",
    "sub_issue_type",
    "priority_level",
    "ticket_status",
    "open_timestamp",
    "first_response_time_hours",
    "sla_target_hours",
    "sla_breach_flag"
]

service_requests["duplicate_id_flag"] = np.where(
    service_requests["ticket_id"].duplicated(keep=False),
    "Y",
    "N"
)

service_requests["_quality_rank"] = (
    service_requests["data_quality_flag"]
    .map(quality_rank)
    .fillna(99)
)

service_requests["_critical_missing"] = (
    service_requests[ticket_critical_columns]
    .isna()
    .sum(axis=1)
)

service_requests["record_version"] = pd.to_numeric(
    service_requests["record_version"],
    errors="coerce"
)

In [18]:
service_requests_ranked = service_requests.sort_values(
    by=[
        "ticket_id",
        "_quality_rank",
        "_critical_missing",
        "record_version",
        "audit_timestamp"
    ],
    ascending=[
        True,
        True,
        True,
        False,
        False
    ],
    na_position="last"
)

service_requests_clean = (
    service_requests_ranked
    .drop_duplicates(
        subset=["ticket_id"],
        keep="first"
    )
    .copy()
)

In [19]:
ticket_duplicates_removed = (
    len(service_requests) - len(service_requests_clean)
)

print(
    "Service Requests before deduplication:",
    len(service_requests)
)
print(
    "Service Requests after deduplication:",
    len(service_requests_clean)
)
print(
    "Duplicate ticket rows removed:",
    ticket_duplicates_removed
)
print(
    "Unique ticket IDs:",
    service_requests_clean["ticket_id"].nunique()
)

Service Requests before deduplication: 4449
Service Requests after deduplication: 4377
Duplicate ticket rows removed: 72
Unique ticket IDs: 4377


In [20]:
print(
    "Duplicated booking IDs remaining:",
    bookings_clean["booking_id"].duplicated().sum()
)

print(
    "Duplicated ticket IDs remaining:",
    service_requests_clean["ticket_id"].duplicated().sum()
)

Duplicated booking IDs remaining: 0
Duplicated ticket IDs remaining: 0


In [21]:
# Mark selected booking records
booking_selected_index = set(bookings_clean.index)

booking_dedup_review = bookings_ranked[
    bookings_ranked["duplicate_id_flag"] == "Y"
].copy()

booking_dedup_review["dedup_result"] = np.where(
    booking_dedup_review.index.isin(booking_selected_index),
    "Kept",
    "Removed"
)

booking_dedup_review[
    [
        "booking_id",
        "dedup_result",
        "record_version",
        "data_quality_flag",
        "_quality_rank",
        "_critical_missing",
        "audit_timestamp",
        "customer_id",
        "property_id",
        "total_revenue"
    ]
].head(30)

,booking_id,dedup_result,record_version,data_quality_flag,_quality_rank,_critical_missing,audit_timestamp,customer_id,property_id,total_revenue
112,BKG_000063,Kept,2,Date Format Issue,1,0,2025-03-03 18:49:00,CUST_00546,P020,"3,358.23"
62,BKG_000063,Removed,1,Potential Test Record,5,0,2024-10-12 03:43:00,CUST_00350,P043,0.00
63,BKG_000064,Kept,2,Potential Test Record,5,0,2025-05-09 18:41:00,CUST_01198,P018,0.00
104,BKG_000064,Removed,1,Potential Test Record,5,0,2025-03-08 06:21:00,CUST_00256,P031,254.55
165,BKG_000142,Kept,2,Date Format Issue,1,0,2025-04-11 04:29:00,CUST_00865,P006,0.00
141,BKG_000142,Removed,2,Manual Review,3,0,2025-05-11 10:20:00,CUST_00110,P010,"1,349.14"
217,BKG_000218,Kept,2,OK,0,0,2024-11-21 17:18:00,CUST_00468,P026,746.60
262,BKG_000218,Removed,1,Date Format Issue,1,0,2025-04-13 09:47:00,CUST_00758,P034,"1,274.21"
305,BKG_000252,Kept,1,Date Format Issue,1,0,2025-03-18 05:20:00,CUST_00396,P011,"3,839.10"
251,BKG_000252,Removed,1,Duplicate-Like,2,0,2024-10-13 02:08:00,CUST_00469,P032,"2,760.82"


In [22]:
# Mark selected service-request records
ticket_selected_index = set(service_requests_clean.index)

ticket_dedup_review = service_requests_ranked[
    service_requests_ranked["duplicate_id_flag"] == "Y"
].copy()

ticket_dedup_review["dedup_result"] = np.where(
    ticket_dedup_review.index.isin(ticket_selected_index),
    "Kept",
    "Removed"
)

ticket_dedup_review[
    [
        "ticket_id",
        "dedup_result",
        "record_version",
        "data_quality_flag",
        "_quality_rank",
        "_critical_missing",
        "audit_timestamp",
        "booking_id",
        "customer_id",
        "property_id",
        "issue_type"
    ]
].head(30)

,ticket_id,dedup_result,record_version,data_quality_flag,_quality_rank,_critical_missing,audit_timestamp,booking_id,customer_id,property_id,issue_type
159,TKT_000160,Kept,2,OK,0,0,2025-01-19 10:53:00,BKG_002388,CUST_00542,P056,Noise Complaint
205,TKT_000160,Removed,1,OK,0,0,2024-10-05 08:49:00,BKG_003022,C-00179,P014,Billing Question
269,TKT_000270,Kept,1,Duplicate-Like,2,0,2025-04-12 23:24:00,BKG_003093,CUST_00150,P050,WiFi Issue
301,TKT_000270,Removed,2,Duplicate-Like,2,1,2025-01-18 13:36:00,NaN,CUST_00771,P016,Housekeeping Delay
420,TKT_000329,Kept,3,OK,0,0,2024-12-19 10:28:00,BKG_000934,CUST_01219,P010,Maintenance
328,TKT_000329,Removed,3,OK,0,0,2024-11-15 23:51:00,BKG_000042,CUST_00253,P042,Check-in Delay
337,TKT_000338,Kept,2,Potential Test Record,5,0,2025-05-05 13:03:00,BKG_003412,CUST_01359,P049,Check-in Delay
412,TKT_000338,Removed,2,Potential Test Record,5,0,2025-03-21 23:25:00,BKG_001594,CUST_00815,P042,Food & Beverage
427,TKT_000428,Kept,3,OK,0,0,2025-03-23 14:10:00,BKG_003483,cust_01147,P036,Housekeeping Delay
437,TKT_000428,Removed,3,Missing Field,4,0,2025-03-20 14:19:00,BKG_001344,CUST_00128,P010,Food & Beverage


In [23]:
print("Selected booking records:")
print(
    booking_dedup_review
    .query("dedup_result == 'Kept'")
    ["data_quality_flag"]
    .value_counts(dropna=False)
)

print("\nSelected service-request records:")
print(
    ticket_dedup_review
    .query("dedup_result == 'Kept'")
    ["data_quality_flag"]
    .value_counts(dropna=False)
)

Selected booking records:
data_quality_flag
OK                       33
Date Format Issue        11
Duplicate-Like            5
Manual Review             5
Potential Test Record     3
Missing Field             2
Name: count, dtype: int64

Selected service-request records:
data_quality_flag
OK                       38
Duplicate-Like           14
Date Format Issue        10
Missing Field             5
Potential Test Record     2
Manual Review             2
Name: count, dtype: int64


In [24]:
booking_version_check = (
    booking_dedup_review
    .groupby("booking_id")
    .agg(
        selected_version=(
            "record_version",
            lambda x: booking_dedup_review.loc[
                x.index
            ].query("dedup_result == 'Kept'")[
                "record_version"
            ].iloc[0]
        ),
        maximum_version=("record_version", "max")
    )
)

booking_version_check[
    booking_version_check["selected_version"]
    < booking_version_check["maximum_version"]
].head(20)

,selected_version,maximum_version
booking_id,,
BKG_001157,1,2
BKG_001546,1,2
BKG_001568,1,2
BKG_001717,1,2
BKG_001820,1,2
BKG_002345,1,2
BKG_002377,1,2
BKG_003264,1,2


In [25]:
ticket_version_check = (
    ticket_dedup_review
    .groupby("ticket_id")
    .agg(
        selected_version=(
            "record_version",
            lambda x: ticket_dedup_review.loc[
                x.index
            ].query("dedup_result == 'Kept'")[
                "record_version"
            ].iloc[0]
        ),
        maximum_version=("record_version", "max")
    )
)

ticket_version_check[
    ticket_version_check["selected_version"]
    < ticket_version_check["maximum_version"]
].head(20)

,selected_version,maximum_version
ticket_id,,
TKT_000270,1,2
TKT_000510,2,3
TKT_000727,1,2
TKT_000829,1,2
TKT_000906,1,3
TKT_000980,1,3
TKT_001273,1,2
TKT_001691,1,2
TKT_001695,1,2


In [27]:
temporary_columns = [
    "_quality_rank",
    "_critical_missing"
]

bookings_clean = bookings_clean.drop(
    columns=temporary_columns
)

service_requests_clean = service_requests_clean.drop(
    columns=temporary_columns
)

# Customers did not contain duplicated non-null customer IDs
customers_clean = customers.copy()

In [28]:
for name, df, key in [
    ("Customers", customers_clean, "customer_id"),
    ("Bookings", bookings_clean, "booking_id"),
    (
        "Service Requests",
        service_requests_clean,
        "ticket_id"
    )
]:
    print(f"\n{name}: {df.shape}")
    print(f"Missing primary keys: {df[key].isna().sum()}")
    print(f"Duplicated primary keys: {df[key].duplicated().sum()}")


Customers: (1474, 50)
Missing primary keys: 0
Duplicated primary keys: 0

Bookings: (3406, 65)
Missing primary keys: 0
Duplicated primary keys: 0

Service Requests: (4377, 61)
Missing primary keys: 0
Duplicated primary keys: 0


In [29]:
def clean_date_columns(df, date_columns):
    df = df.copy()

    for column in date_columns:
        original_non_null = df[column].notna()

        parsed_date = pd.to_datetime(
            df[column],
            errors="coerce",
            format="mixed"
        )

        # Flag values that existed but could not be parsed
        df[f"{column}_invalid_flag"] = np.where(
            original_non_null & parsed_date.isna(),
            "Y",
            "N"
        )

        df[column] = parsed_date

        invalid_count = (
            df[f"{column}_invalid_flag"] == "Y"
        ).sum()

        print(
            f"{column}: "
            f"{invalid_count} invalid values"
        )

    return df

In [30]:
customer_date_columns = [
    "signup_date",
    "last_booking_date",
    "audit_timestamp",
    "extract_date"
]

customers_clean = clean_date_columns(
    customers_clean,
    customer_date_columns
)

signup_date: 0 invalid values
last_booking_date: 0 invalid values
audit_timestamp: 0 invalid values
extract_date: 0 invalid values


In [31]:
booking_date_columns = [
    "property_open_date",
    "booking_date",
    "check_in_date",
    "check_out_date",
    "audit_timestamp",
    "extract_date"
]

bookings_clean = clean_date_columns(
    bookings_clean,
    booking_date_columns
)

property_open_date: 0 invalid values
booking_date: 0 invalid values
check_in_date: 0 invalid values
check_out_date: 0 invalid values
audit_timestamp: 0 invalid values
extract_date: 0 invalid values


In [32]:
service_date_columns = [
    "property_open_date",
    "open_timestamp",
    "resolved_timestamp",
    "audit_timestamp",
    "extract_date"
]

service_requests_clean = clean_date_columns(
    service_requests_clean,
    service_date_columns
)

property_open_date: 0 invalid values
open_timestamp: 0 invalid values
resolved_timestamp: 0 invalid values
audit_timestamp: 0 invalid values
extract_date: 0 invalid values


In [33]:
print("Customers:")
print(customers_clean[customer_date_columns].dtypes)

print("\nBookings:")
print(bookings_clean[booking_date_columns].dtypes)

print("\nService Requests:")
print(service_requests_clean[service_date_columns].dtypes)

Customers:
signup_date          datetime64[ns]
last_booking_date    datetime64[ns]
audit_timestamp      datetime64[ns]
extract_date         datetime64[ns]
dtype: object

Bookings:
property_open_date    datetime64[ns]
booking_date          datetime64[ns]
check_in_date         datetime64[ns]
check_out_date        datetime64[ns]
audit_timestamp       datetime64[ns]
extract_date          datetime64[ns]
dtype: object

Service Requests:
property_open_date    datetime64[ns]
open_timestamp        datetime64[ns]
resolved_timestamp    datetime64[ns]
audit_timestamp       datetime64[ns]
extract_date          datetime64[ns]
dtype: object


In [34]:
bookings_clean["invalid_stay_date_flag"] = np.where(
    bookings_clean["check_out_date"]
    <= bookings_clean["check_in_date"],
    "Y",
    "N"
)

bookings_clean["invalid_booking_date_flag"] = np.where(
    bookings_clean["booking_date"]
    > bookings_clean["check_in_date"],
    "Y",
    "N"
)

print(
    "Check-out on or before check-in:",
    (
        bookings_clean["invalid_stay_date_flag"]
        == "Y"
    ).sum()
)

print(
    "Booking date after check-in:",
    (
        bookings_clean["invalid_booking_date_flag"]
        == "Y"
    ).sum()
)

Check-out on or before check-in: 0
Booking date after check-in: 0


In [35]:
service_requests_clean[
    "invalid_resolution_date_flag"
] = np.where(
    service_requests_clean["resolved_timestamp"].notna()
    & (
        service_requests_clean["resolved_timestamp"]
        < service_requests_clean["open_timestamp"]
    ),
    "Y",
    "N"
)

print(
    "Resolution before ticket opening:",
    (
        service_requests_clean[
            "invalid_resolution_date_flag"
        ] == "Y"
    ).sum()
)

Resolution before ticket opening: 0


In [36]:
bookings_clean["calculated_length_of_stay"] = (
    bookings_clean["check_out_date"]
    - bookings_clean["check_in_date"]
).dt.days

bookings_clean["calculated_lead_time_days"] = (
    bookings_clean["check_in_date"]
    - bookings_clean["booking_date"]
).dt.days

In [37]:
bookings_clean["length_of_stay_difference"] = (
    bookings_clean["length_of_stay"]
    - bookings_clean["calculated_length_of_stay"]
)

bookings_clean["lead_time_difference"] = (
    bookings_clean["lead_time_days"]
    - bookings_clean["calculated_lead_time_days"]
)

In [38]:
bookings_clean["length_of_stay_mismatch_flag"] = np.where(
    bookings_clean["length_of_stay"].notna()
    & bookings_clean["calculated_length_of_stay"].notna()
    & (
        bookings_clean["length_of_stay_difference"]
        .abs() > 0.01
    ),
    "Y",
    "N"
)

bookings_clean["lead_time_mismatch_flag"] = np.where(
    bookings_clean["lead_time_days"].notna()
    & bookings_clean["calculated_lead_time_days"].notna()
    & (
        bookings_clean["lead_time_difference"]
        .abs() > 0.01
    ),
    "Y",
    "N"
)

print(
    "Length-of-stay mismatches:",
    (
        bookings_clean["length_of_stay_mismatch_flag"]
        == "Y"
    ).sum()
)

print(
    "Lead-time mismatches:",
    (
        bookings_clean["lead_time_mismatch_flag"]
        == "Y"
    ).sum()
)

Length-of-stay mismatches: 0
Lead-time mismatches: 0


In [39]:
service_requests_clean[
    "calculated_resolution_time_hours"
] = (
    service_requests_clean["resolved_timestamp"]
    - service_requests_clean["open_timestamp"]
).dt.total_seconds() / 3600

In [40]:
service_requests_clean[
    "resolution_time_difference"
] = (
    service_requests_clean["resolution_time_hours"]
    - service_requests_clean[
        "calculated_resolution_time_hours"
    ]
)

service_requests_clean[
    "resolution_time_mismatch_flag"
] = np.where(
    service_requests_clean[
        "resolution_time_hours"
    ].notna()
    & service_requests_clean[
        "calculated_resolution_time_hours"
    ].notna()
    & (
        service_requests_clean[
            "resolution_time_difference"
        ].abs() > 0.1
    ),
    "Y",
    "N"
)

print(
    "Resolution-time mismatches:",
    (
        service_requests_clean[
            "resolution_time_mismatch_flag"
        ] == "Y"
    ).sum()
)

Resolution-time mismatches: 0


In [41]:
print(
    "Missing original length of stay:",
    bookings_clean["length_of_stay"].isna().sum()
)

print(
    "Missing calculated length of stay:",
    bookings_clean[
        "calculated_length_of_stay"
    ].isna().sum()
)

print(
    "Missing original lead time:",
    bookings_clean["lead_time_days"].isna().sum()
)

print(
    "Missing calculated lead time:",
    bookings_clean[
        "calculated_lead_time_days"
    ].isna().sum()
)

print(
    "Missing original resolution time:",
    service_requests_clean[
        "resolution_time_hours"
    ].isna().sum()
)

print(
    "Missing calculated resolution time:",
    service_requests_clean[
        "calculated_resolution_time_hours"
    ].isna().sum()
)

Missing original length of stay: 56
Missing calculated length of stay: 0
Missing original lead time: 50
Missing calculated lead time: 0
Missing original resolution time: 1127
Missing calculated resolution time: 1127


In [42]:
bookings_clean["analysis_length_of_stay"] = np.where(
    bookings_clean["calculated_length_of_stay"] > 0,
    bookings_clean["calculated_length_of_stay"],
    bookings_clean["length_of_stay"]
)

bookings_clean["analysis_lead_time_days"] = np.where(
    bookings_clean["calculated_lead_time_days"] >= 0,
    bookings_clean["calculated_lead_time_days"],
    bookings_clean["lead_time_days"]
)

In [46]:
service_requests_clean[
    "analysis_resolution_time_hours"
] = np.where(
    service_requests_clean[
        "calculated_resolution_time_hours"
    ] >= 0,
    service_requests_clean[
        "calculated_resolution_time_hours"
    ],
    service_requests_clean[
        "resolution_time_hours"
    ]
)

In [47]:
unresolved_statuses = [
    "Open",
    "Escalated",
    "Pending Customer"
]

service_requests_clean.loc[
    service_requests_clean["ticket_status"].isin(
        unresolved_statuses
    ),
    "analysis_resolution_time_hours"
] = np.nan

In [49]:
# Create an empty string column first
service_requests_clean[
    "calculated_sla_breach_flag"
] = pd.Series(
    pd.NA,
    index=service_requests_clean.index,
    dtype="string"
)

# Records with valid resolution time and SLA target
valid_sla_mask = (
    service_requests_clean[
        "analysis_resolution_time_hours"
    ].notna()
    & service_requests_clean[
        "sla_target_hours"
    ].notna()
)

# Assign Y or N only to valid records
service_requests_clean.loc[
    valid_sla_mask,
    "calculated_sla_breach_flag"
] = np.where(
    service_requests_clean.loc[
        valid_sla_mask,
        "analysis_resolution_time_hours"
    ]
    > service_requests_clean.loc[
        valid_sla_mask,
        "sla_target_hours"
    ],
    "Y",
    "N"
)

In [50]:
service_requests_clean[
    "calculated_sla_breach_flag"
].value_counts(dropna=False)

calculated_sla_breach_flag
N       1743
<NA>    1399
Y       1235
Name: count, dtype: Int64

In [51]:
sla_comparison_mask = (
    service_requests_clean[
        "calculated_sla_breach_flag"
    ].notna()
    & service_requests_clean[
        "sla_breach_flag"
    ].notna()
)